# ml_0 : le socle du machine learning

**But** : mesurer, sur les clients laissés au machine learning, ce que les **6 modèles d'origine** obtiennent avec les **23 variables d'origine** du dataset, sans aucune colonne créée. C'est le point zéro : chaque scénario suivant (`ml_1`, `ml_2`…) ajoute des colonnes et se compare à ce socle.

**Données** : `data/ML/train_dataset.csv`, produit par `creation_datasets_ML.ipynb` (périmètre S12, clients au contentieux à M retirés, 80 % des clients restants). **Le jeu de test n'est pas lu** : il est réservé à l'évaluation finale, une seule fois, une fois le modèle figé.

**Méthode** (identique pour tous les scénarios) :
- validation croisée stratifiée à 5 plis, `random_state=42`, sur le train uniquement ;
- **score de décision** = moyenne du F2 et du ROC AUC, mesurés en validation ; il est donné avec son **écart-type sur les 5 plis** : un écart entre deux modèles plus petit que cet écart-type n'est pas une différence, c'est du bruit ;
- seuil de décision à 0,5 (valeur par défaut des modèles) : le F2, les défauts détectés et la précision en dépendent ; le seuil sera réglé sur le modèle final ;
- défauts détectés et précision mesurés en validation (chaque client est prédit par un modèle qui ne l'a pas vu).

**Étapes** :
1. chargement du train et choix des variables ;
2. premier essai des 6 modèles, avec leurs réglages par défaut ;
3. recherche des hyperparamètres par une boucle automatique de GridSearch : choix du meilleur score de décision sous contrainte de surapprentissage, grille corrigée tour après tour (paramètres au bord, surapprentissage) ;
4. matrices de confusion (versions optimisées sur le ROC AUC et sur le F2), défauts détectés, précision et importance des variables, en validation ;
5. podium détaillé et enregistrement du meilleur modèle.

In [ ]:
numero_scenario = "0"

## 1. Chargement et variables

Le train est lu, puis la cellule **« variables du scénario »** fixe les colonnes données aux modèles. C'est la seule cellule à modifier pour changer de variables d'un scénario à l'autre.

**Le socle** : les 23 variables d'origine, c'est-à-dire le plafond (`LIMIT_BAL`), la démographie (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`), les codifications de paiement (`PAY_1` à `PAY_6`, corrigées au niveau 5 du nettoyage), les factures (`BILL_AMT1` à `BILL_AMT6`) et les paiements (`PAY_AMT1` à `PAY_AMT6`). L'identifiant `ID` et la cible `dpnm` ne sont jamais des variables.

**Ajouter une colonne** (elle doit être définie dans `docs/colonnes_creees.md` et présente dans le train) : la ranger dans la liste qui correspond à son encodage :
- `ajout_num` : nombres, montants, ratios, compteurs, indicateurs 0 / 1 ;
- `ajout_cat` : catégories sans ordre (une colonne par catégorie), par exemple un type d'usage ;
- `ajout_ord` : catégories dans un ordre naturel (encodage ordinal), par exemple une codification.

**Retirer une colonne du socle** (variante sans codifications, par exemple) : l'écrire dans `retirees`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

train = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
print(f"Clients du train : {len(train)} ; colonnes disponibles : {train.shape[1]}")

In [ ]:
# ==============================================================================
# VARIABLES DU SCÉNARIO : seule cellule à modifier pour changer les colonnes
# ==============================================================================

# Colonnes ajoutées par ce scénario, rangées selon leur encodage
ajout_num = []   # nombres, montants, ratios, compteurs, indicateurs 0 / 1
ajout_cat = []   # catégories sans ordre (une colonne par catégorie)
ajout_ord = []   # catégories dans un ordre naturel (encodage ordinal)

# Colonnes du socle retirées par ce scénario
retirees = []

# ------------------------------------------------------------------------------
# Socle : les 23 variables d'origine (ne pas modifier)
socle_num = ["LIMIT_BAL", "AGE"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
socle_cat = ["SEX", "EDUCATION", "MARRIAGE"]
socle_ord = [f"PAY_{n}" for n in range(1, 7)]

num_col = [c for c in socle_num if c not in retirees] + ajout_num
cat_col = [c for c in socle_cat if c not in retirees] + ajout_cat
ord_col = [c for c in socle_ord if c not in retirees] + ajout_ord
variables = num_col + cat_col + ord_col

# Contrôles : colonnes présentes dans le train, aucune en double, ni identifiant ni cible
absentes = [c for c in variables + retirees if c not in train.columns]
doublons = sorted({c for c in variables if variables.count(c) > 1})
interdites = [c for c in variables if c in ("ID", "dpnm")]
assert not absentes, f"Colonnes absentes du train : {absentes} (à créer dans creation_datasets_ML.ipynb)"
assert not doublons, f"Colonnes en double : {doublons}"
assert not interdites, f"Colonnes interdites comme variables : {interdites}"

X_train = train[variables].reset_index(drop=True)
y_train = train["dpnm"].reset_index(drop=True)

print(f"Variables        : {len(variables)} ({len(num_col)} numériques, {len(cat_col)} catégorielles, {len(ord_col)} ordinales)")
print(f"Ajoutées         : {ajout_num + ajout_cat + ajout_ord if ajout_num + ajout_cat + ajout_ord else 'aucune'}")
print(f"Retirées         : {retirees if retirees else 'aucune'}")
print(f"Défauts          : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

**Encodage** (dans le pipeline, donc appris sur chaque pli de la validation croisée, sans fuite) :
- variables numériques (`num_col` : montants, plafond, âge et colonnes de `ajout_num`) : centrées et réduites (`StandardScaler`) ;
- catégories sans ordre (`cat_col` : `SEX`, `EDUCATION`, `MARRIAGE` et colonnes de `ajout_cat`) : une colonne par catégorie (`OneHotEncoder`) ;
- catégories ordonnées (`ord_col` : `PAY_1` à `PAY_6` et colonnes de `ajout_ord`) : encodage ordinal, qui garde l'ordre des valeurs.

Le déséquilibre des classes (environ 1 client sur 6 en défaut) est compensé par une pondération des défauts pour les modèles qui l'acceptent (`class_weight='balanced'`, `scale_pos_weight` pour CatBoost). KNN et le réseau de neurones (MLP) n'ont pas cette option : ils sont gardés tels quels, comme dans la première itération.

In [ ]:
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, make_scorer, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_col),
    ("nom", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cat_col),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ord_col),
])

def score_decision(estimator, X, y):
    # Score de décision : moyenne du F2 (au seuil de 0,5) et du ROC AUC
    proba = estimator.predict_proba(X)[:, 1]
    pred = estimator.predict(X)
    return (fbeta_score(y, pred, beta=2) + roc_auc_score(y, proba)) / 2

scorers = {
    "decision": score_decision,
    "f2": make_scorer(fbeta_score, beta=2),
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles lancés dans ce notebook : retirer un nom pour ne pas le lancer (le reste du notebook s'adapte)
MODELES_A_TESTER = ["LogisticRegression", "SVM", "MLPClassifier", "KNN", "RandomForest", "CatBoost"]

def modeles_de_base():
    # Les 6 modèles d'origine, avec leurs réglages de la première itération ; seuls ceux de MODELES_A_TESTER sont rendus
    tous = {
        "LogisticRegression": make_pipeline(preprocessor, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "SVM": make_pipeline(preprocessor, CalibratedClassifierCV(estimator=SVC(class_weight="balanced", random_state=42), ensemble=False)),
        "MLPClassifier": make_pipeline(preprocessor, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "KNN": make_pipeline(preprocessor, KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
        "RandomForest": make_pipeline(preprocessor, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(preprocessor, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

SEUIL_SURAPPRENTISSAGE = 0.05   # écart train - val du score de décision au-delà duquel on alerte
METRIQUES = {"decision": "Score de décision", "f2": "F2", "roc_auc": "ROC AUC", "recall": "Rappel", "precision": "Précision"}

def alerte(ecart):
    if ecart > 2 * SEUIL_SURAPPRENTISSAGE:
        return "⚠️⚠️ SURAPPRENTISSAGE FORT"
    if ecart > SEUIL_SURAPPRENTISSAGE:
        return "⚠️ SURAPPRENTISSAGE"
    return "✅ OK"

def ligne_scores(nom, moy_val, std_val, moy_train):
    # Une ligne de tableau : chaque métrique en entraînement et en validation, l'écart-type du score de décision et l'alerte
    ligne = {"Modèle": nom}
    for cle, libelle in METRIQUES.items():
        ligne[f"{libelle} (train)"] = moy_train[cle]
        ligne[f"{libelle} (val)"] = moy_val[cle]
    ligne["Écart-type décision (val)"] = std_val
    ligne["Écart train - val"] = moy_train["decision"] - moy_val["decision"]
    ligne["Alerte"] = alerte(ligne["Écart train - val"])
    return ligne

def afficher_scores(lignes, titre):
    df_scores = pd.DataFrame(lignes).sort_values("Score de décision (val)", ascending=False).reset_index(drop=True)
    print(f"=== {titre} ===")
    print(df_scores.drop(columns=[c for c in df_scores.columns if c.startswith("Meilleurs") or c.startswith("Bord")]).round(4).to_string(index=False))
    for _, l in df_scores[df_scores["Écart train - val"] > SEUIL_SURAPPRENTISSAGE].iterrows():
        print(f"{l['Alerte']} : {l['Modèle']}, écart train - val de {l['Écart train - val']:.4f}")
    return df_scores

## 2. Premier essai des 6 modèles (réglages par défaut)

Chaque modèle est entraîné et évalué sur les 5 plis, avec ses réglages par défaut. Le tableau donne chaque métrique en **entraînement** et en **validation**, l'écart-type du score de décision sur les plis, l'écart entre entraînement et validation et une alerte de surapprentissage.

In [ ]:
from sklearn.model_selection import cross_validate

lignes = []
for nom, modele in modeles_de_base().items():
    res = cross_validate(modele, X_train, y_train, cv=cv, scoring=scorers, return_train_score=True, n_jobs=-1, error_score="raise")
    moy_val = {k: res[f"test_{k}"].mean() for k in METRIQUES}
    moy_train = {k: res[f"train_{k}"].mean() for k in METRIQUES}
    lignes.append(ligne_scores(nom, moy_val, res["test_decision"].std(), moy_train))

df_essai = afficher_scores(lignes, "PREMIER ESSAI, RÉGLAGES PAR DÉFAUT")

**Lecture** : une ligne par modèle, triées par score de décision en validation. Seule la validation compte pour comparer les modèles ; l'entraînement sert à mesurer le surapprentissage.
- **Écart-type** : variation d'un pli à l'autre ; deux modèles dont les scores diffèrent de moins que cet écart-type sont à égalité.
- **Écart train - val** : au-delà de 0,05, le modèle apprend par cœur les clients d'entraînement sans mieux prédire les autres (⚠️) ; au-delà de 0,10, le surapprentissage est fort (⚠️⚠️).

## 3. Recherche des hyperparamètres : boucle automatique

Pour chaque modèle, la recherche se fait en **tours** successifs, sur le train uniquement (validation croisée à 5 plis) :
1. **une GridSearch** sur la grille du tour ; le premier tour part des grilles de la première itération ;
2. **un choix sous contrainte de surapprentissage** : parmi les réglages dont l'écart train - val du score de décision reste sous le seuil (0,05), on garde celui qui a le meilleur score de décision en validation ; si aucun ne passe, celui qui surapprend le moins ;
3. **une correction automatique de la grille** pour le tour suivant :
   - **paramètre au bord** de sa grille (la meilleure valeur est la plus petite ou la plus grande testée) : la grille est prolongée de ce côté ;
   - **surapprentissage** : chaque paramètre est poussé dans le sens qui simplifie le modèle (arbres moins profonds, feuilles plus grosses, régularisation plus forte…) ;
   - les autres paramètres sont fixés à leur meilleure valeur, pour que la grille reste petite ;
   - chaque paramètre reste dans des bornes fixées à l'avance (tableau `REGLAGES_PARAMETRES`).

**Arrêt pour un modèle**, au premier de ces cas : plus de surapprentissage ni de paramètre au bord (modèle optimisé) ; un tour qui ne gagne pas plus que l'écart-type (stagnation) ; une grille qui ne peut plus bouger (bornes atteintes) ; `TOURS_MAX` tours atteints.
Un modèle arrêté sort de la boucle : la version retenue, déjà entraînée sur tout le train par la GridSearch, est gardée telle quelle, sans réentraînement. **La version retenue est celle du meilleur tour** : d'abord les tours sans surapprentissage, puis le meilleur score de décision ; à score égal (écart plus petit que l'écart-type), celui qui surapprend le moins.

Plus on teste de réglages, plus le score de validation devient optimiste (un réglage peut avoir eu de la chance sur ces 5 plis) : le nombre de tours est donc limité, et le test, lu une seule fois à la fin, reste le vrai juge.

In [ ]:
from sklearn.model_selection import GridSearchCV

TOURS_MAX = 4   # nombre maximal de tours par modèle (sécurité : temps de calcul et optimisme du score de validation)

# Grilles du premier tour : celles de la première itération
grilles = {
    "LogisticRegression": {"logisticregression__C": [0.01, 0.1, 1.0, 10.0]},
    "SVM": {"calibratedclassifiercv__estimator__C": [1, 2, 3], "calibratedclassifiercv__estimator__gamma": ["scale", "auto"]},
    "MLPClassifier": {"mlpclassifier__hidden_layer_sizes": [(32, 16), (64, 32), (32,)], "mlpclassifier__alpha": [0.01, 0.1, 1.0]},
    "KNN": {"kneighborsclassifier__n_neighbors": [15, 25, 37, 51], "kneighborsclassifier__weights": ["uniform", "distance"]},
    "RandomForest": {"randomforestclassifier__max_depth": [6, 8, 10], "randomforestclassifier__min_samples_leaf": [20, 50, 100], "randomforestclassifier__n_estimators": [200, 300]},
    "CatBoost": {"catboostclassifier__depth": [4, 6], "catboostclassifier__l2_leaf_reg": [5, 10, 15], "catboostclassifier__learning_rate": [0.03, 0.05]},
}

# Comment faire évoluer chaque paramètre numérique d'un tour à l'autre :
# - simplifier : sens qui réduit le surapprentissage (+1 = augmenter, -1 = diminuer, 0 = sans effet sur le surapprentissage) ;
# - pas : écart entre deux valeurs (« x » = multiplier, « + » = ajouter) ;
# - bornes : valeurs minimale et maximale autorisées.
# Les paramètres absents de ce tableau (catégories comme gamma ou weights, nombre d'arbres) sont fixés à leur meilleure valeur après le premier tour.
REGLAGES_PARAMETRES = {
    "logisticregression__C":                    {"simplifier": -1, "pas": ("x", 3),   "bornes": (1e-4, 1e4),  "entier": False},
    "calibratedclassifiercv__estimator__C":     {"simplifier": -1, "pas": ("x", 2),   "bornes": (0.01, 100),  "entier": False},
    "mlpclassifier__alpha":                     {"simplifier": +1, "pas": ("x", 3),   "bornes": (1e-5, 100),  "entier": False},
    "kneighborsclassifier__n_neighbors":        {"simplifier": +1, "pas": ("x", 1.5), "bornes": (3, 401),     "entier": True},
    "randomforestclassifier__max_depth":        {"simplifier": -1, "pas": ("+", 2),   "bornes": (2, 30),      "entier": True},
    "randomforestclassifier__min_samples_leaf": {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 2000),    "entier": True},
    "catboostclassifier__depth":                {"simplifier": -1, "pas": ("+", 1),   "bornes": (2, 10),      "entier": True},
    "catboostclassifier__l2_leaf_reg":          {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 200),     "entier": False},
    "catboostclassifier__learning_rate":        {"simplifier": -1, "pas": ("x", 1.5), "bornes": (0.003, 0.3), "entier": False},
}

def nom_court(parametre):
    return parametre.split("__")[-1]

def arrondir(valeur, entier):
    return int(round(valeur)) if entier else float(f"{valeur:.3g}")

def voisin(valeur, reglage, sens):
    # Valeur suivante dans un sens (+1 = plus grande, -1 = plus petite), dans les bornes
    mode, pas = reglage["pas"]
    if mode == "x":
        nouvelle = valeur * pas if sens > 0 else valeur / pas
    else:
        nouvelle = valeur + pas if sens > 0 else valeur - pas
    bas, haut = reglage["bornes"]
    nouvelle = arrondir(min(max(nouvelle, bas), haut), reglage["entier"])
    return None if nouvelle == valeur else nouvelle

def choisir_index(r):
    # Choix sous contrainte : meilleur score de décision parmi les réglages sans surapprentissage ; sinon le moins surappris
    ecarts = r["mean_train_decision"] - r["mean_test_decision"]
    sous_seuil = np.where(ecarts <= SEUIL_SURAPPRENTISSAGE)[0]
    if len(sous_seuil):
        return int(sous_seuil[np.argmax(r["mean_test_decision"][sous_seuil])])
    return int(np.argmin(ecarts))

def bords_de_grille(grille, meilleurs):
    # Paramètres numériques dont la meilleure valeur est la plus petite ou la plus grande de la grille (+1 = maximum, -1 = minimum)
    bords = {}
    for p, valeurs in grille.items():
        if p in REGLAGES_PARAMETRES and len(valeurs) > 1:
            if meilleurs[p] == max(valeurs):
                bords[p] = +1
            elif meilleurs[p] == min(valeurs):
                bords[p] = -1
    return bords

def grille_suivante(grille, meilleurs, bords, surapprend):
    # Grille du tour suivant et explication des corrections ; None si plus rien ne peut bouger
    nouvelle, actions = {}, []
    for p, valeurs in grille.items():
        v = meilleurs[p]
        reglage = REGLAGES_PARAMETRES.get(p)
        if reglage is None:
            nouvelle[p] = [v]                            # catégorie ou paramètre sans effet : fixé
            continue
        sens = None
        if surapprend and reglage["simplifier"] != 0:
            sens = reglage["simplifier"]                 # pousser vers un modèle plus simple
            raison = "surapprentissage"
        elif p in bords:
            sens = bords[p]                              # prolonger la grille du côté du bord
            raison = "bord de grille"
        if sens is None:
            nouvelle[p] = [v]
            continue
        v1 = voisin(v, reglage, sens)
        v2 = voisin(v1, reglage, sens) if v1 is not None else None
        valeurs_tour = [x for x in (v, v1, v2) if x is not None]
        nouvelle[p] = sorted(set(valeurs_tour))
        if len(nouvelle[p]) > 1:
            actions.append(f"{nom_court(p)} {v} → {nouvelle[p]} ({raison})")
    if not actions:
        return None, ["grille bloquée (bornes atteintes)"]
    return nouvelle, actions

def lancer_grille(nom, modele, grille):
    grid = GridSearchCV(modele, grille, cv=cv, scoring=scorers, refit=choisir_index, return_train_score=True, n_jobs=-1)
    grid.fit(X_train, y_train)
    r, i = grid.cv_results_, grid.best_index_
    ligne = ligne_scores(nom,
                         {k: r[f"mean_test_{k}"][i] for k in METRIQUES},
                         r["std_test_decision"][i],
                         {k: r[f"mean_train_{k}"][i] for k in METRIQUES})
    ligne["Meilleurs paramètres"] = {nom_court(k): v for k, v in grid.best_params_.items()}
    return grid, ligne

def meilleur_tour(a, b):
    # Compare deux tours : d'abord sans surapprentissage, puis meilleur score ; à égalité (dans l'écart-type), le moins surappris
    a_ok, b_ok = a["Écart train - val"] <= SEUIL_SURAPPRENTISSAGE, b["Écart train - val"] <= SEUIL_SURAPPRENTISSAGE
    if a_ok != b_ok:
        return a if a_ok else b
    if abs(a["Score de décision (val)"] - b["Score de décision (val)"]) <= max(a["Écart-type décision (val)"], b["Écart-type décision (val)"]):
        return a if a["Écart train - val"] <= b["Écart train - val"] else b
    return a if a["Score de décision (val)"] > b["Score de décision (val)"] else b

# ------------------------------------------------------------------------------
# Boucle
best_models, recherches, lignes_retenues, historique = {}, {}, [], []
for nom, modele in modeles_de_base().items():
    grille, retenu, fin = grilles[nom], None, None
    for tour in range(1, TOURS_MAX + 1):
        grid, ligne = lancer_grille(nom, modele, grille)
        ligne["Tour"] = tour
        surapprend = ligne["Écart train - val"] > SEUIL_SURAPPRENTISSAGE
        bords = bords_de_grille(grille, grid.best_params_)
        precedent = retenu
        if retenu is None or meilleur_tour(retenu[1], ligne) is ligne:
            retenu = (grid, ligne)
        grille_tour = {nom_court(p): v for p, v in grille.items()}
        if not surapprend and not bords:
            fin = "optimisé : ni surapprentissage ni paramètre au bord"
        elif precedent is not None and not surapprend and ligne["Score de décision (val)"] - precedent[1]["Score de décision (val)"] <= precedent[1]["Écart-type décision (val)"]:
            fin = "stagnation : gain plus petit que l'écart-type"
        elif tour == TOURS_MAX:
            fin = "nombre maximal de tours atteint"
        else:
            grille, actions = grille_suivante(grille, grid.best_params_, bords, surapprend)
            if grille is None:
                fin = actions[0]
        historique.append({
            "Modèle": nom, "Tour": tour, "Grille testée": grille_tour, "Réglage choisi": ligne["Meilleurs paramètres"],
            "Décision (val)": ligne["Score de décision (val)"], "Écart-type": ligne["Écart-type décision (val)"],
            "Écart train - val": ligne["Écart train - val"], "Alerte": ligne["Alerte"],
            "Au bord": [f"{nom_court(p)} ({'maximum' if s > 0 else 'minimum'})" for p, s in bords.items()],
            "Suite": fin if fin else "; ".join(actions),
        })
        if fin:
            break
    best_models[nom], recherches[nom] = retenu[0].best_estimator_, retenu[0]
    lignes_retenues.append(dict(retenu[1], **{"Arrêt": fin}))
    print(f"{nom} : {tour} tour(s), {fin} ; version retenue : tour {retenu[1]['Tour']}")

print()
df_final = afficher_scores(lignes_retenues, "VERSION RETENUE DE CHAQUE MODÈLE")
print("\n=== HISTORIQUE DE LA BOUCLE ===")
print(pd.DataFrame(historique).round(4).to_string(index=False))

**Lecture** :
- les lignes sous la cellule donnent, pour chaque modèle, le nombre de tours, la raison de l'arrêt et le tour retenu ;
- le premier tableau donne les scores de la version retenue de chaque modèle (même présentation qu'à l'étape 2) ;
- l'historique donne, tour par tour, la grille testée, le réglage choisi, son score, son surapprentissage, les paramètres au bord, puis la correction appliquée au tour suivant (ou la raison de l'arrêt). Un modèle qui reste en surapprentissage après tous ses tours est signalé : sa grille de départ ou ses bornes sont peut-être à revoir.

## 4. Matrices de confusion, défauts détectés et importance des variables, en validation

Une matrice de confusion n'affiche pas un score : elle compte les prédictions d'un modèle au seuil de 0,5. Elle dépend donc des réglages du modèle, et ces réglages dépendent de la métrique optimisée. La GridSearch du tour retenu a calculé le F2 et le ROC AUC de chaque combinaison : on retrouve sans nouvelle recherche, pour chaque modèle, **la meilleure version selon le ROC AUC** et **la meilleure version selon le F2**, comme les deux optimisations de la première itération. Ces deux versions sont choisies sans la contrainte de surapprentissage : elles peuvent surapprendre davantage. La version retenue pour le podium reste celle de la boucle (score de décision, sous contrainte).

Chaque client du train est prédit par un modèle entraîné sur les 4 autres plis, donc sans l'avoir vu (`cross_val_predict`).
- **Matrices de confusion** (une série pour le ROC AUC, une pour le F2) : en ligne, la réalité (solvable, défaut) ; en colonne, la prédiction. Chaque case donne l'effectif et la part de la ligne : la case en bas à droite est la part des défauts détectés (rappel).
- **Tableau** : pour chaque modèle et chaque optimisation (score de décision, ROC AUC, F2), les réglages, les prédits en défaut, les défauts détectés, les fausses alertes, les défauts manqués, la précision (part des clients signalés réellement en défaut) et le rappel.
- **Importance des variables** : poids de chaque variable dans la version retenue (score de décision), entraînée sur tout le train, en pourcentage (importance des arbres, coefficients de la régression logistique, poids de la première couche du réseau de neurones ; SVM et KNN n'en ont pas).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import cross_val_predict

ordre = df_final["Modèle"].tolist()
OPTIMISATIONS = {"decision": "score de décision", "roc_auc": "ROC AUC", "f2": "F2"}

def reglages(nom, metrique):
    # Score de décision : la version retenue par la boucle (choix sous contrainte de surapprentissage) ;
    # ROC AUC et F2 : la meilleure combinaison du tour retenu selon cette métrique, sans contrainte
    if metrique == "decision":
        return recherches[nom].best_params_
    r = recherches[nom].cv_results_
    return r["params"][int(np.argmin(r[f"rank_test_{metrique}"]))]

predictions = {}   # même réglage = même prédiction : calculée une seule fois
def predire(nom, params):
    cle = (nom, str(sorted(params.items())))
    if cle not in predictions:
        modele = clone(modeles_de_base()[nom]).set_params(**params)
        predictions[cle] = cross_val_predict(modele, X_train, y_train, cv=cv, n_jobs=-1)
    return predictions[cle]

lignes = []
for metrique, libelle in OPTIMISATIONS.items():
    afficher = metrique != "decision"
    if afficher:
        nb_lignes = (len(ordre) + 1) // 2
        fig, axes = plt.subplots(nb_lignes, 2, figsize=(12, 5 * nb_lignes), squeeze=False)
        for ax_vide in axes.flatten()[len(ordre):]:
            ax_vide.axis("off")
        fig.suptitle(f"Matrices de confusion en validation : modèles optimisés sur le {libelle}", fontsize=15, fontweight="bold")
    for k, nom in enumerate(ordre):
        params = reglages(nom, metrique)
        cm = confusion_matrix(y_train, predire(nom, params))
        tn, fp, fn, tp = cm.ravel()
        lignes.append({
            "Modèle": nom, "Optimisé sur": libelle, "Réglages": {nom_court(p): v for p, v in params.items()},
            "Prédits en défaut": tp + fp, "Défauts détectés": tp, "Fausses alertes": fp, "Défauts manqués": fn,
            "Précision (%)": tp / (tp + fp) * 100 if tp + fp else np.nan, "Rappel (%)": tp / (tp + fn) * 100,
        })
        if afficher:
            ax = axes.flatten()[k]
            cm_part = cm / cm.sum(axis=1, keepdims=True)
            ConfusionMatrixDisplay(cm_part, display_labels=["Solvable (0)", "Défaut (1)"]).plot(cmap="Blues", ax=ax, colorbar=False, include_values=False)
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, f"{cm[i, j]:,}\n({cm_part[i, j]:.1%})".replace(",", " "), ha="center", va="center",
                            color="white" if cm_part[i, j] > 0.5 else "black", fontweight="bold", fontsize=11)
            ax.set_title(f"{nom} (optimisé {libelle})")
    if afficher:
        plt.tight_layout()
        plt.show()

df_detection = pd.DataFrame(lignes).round(2)
print(f"Défauts dans le train : {y_train.sum()} ; taux de défaut (précision du hasard) : {y_train.mean() * 100:.2f} %")
print(df_detection.to_string(index=False))

# Importance des variables
noms_variables = [n.split("__", 1)[1] for n in best_models[ordre[0]][0].get_feature_names_out()]
importances = {}
for nom in ordre:
    clf = best_models[nom][-1]
    if isinstance(clf, CalibratedClassifierCV):
        clf = clf.calibrated_classifiers_[0].estimator
    if hasattr(clf, "feature_importances_"):
        brut = np.asarray(clf.feature_importances_, dtype=float)
    elif hasattr(clf, "coef_"):
        brut = np.abs(clf.coef_[0])
    elif hasattr(clf, "coefs_"):
        brut = np.abs(clf.coefs_[0]).sum(axis=1)
    else:
        continue
    importances[nom] = brut / brut.sum() * 100
df_importances = pd.DataFrame(importances, index=noms_variables)
df_importances["Moyenne (%)"] = df_importances.mean(axis=1)
print("\n=== IMPORTANCE DES VARIABLES (%) ===")
print(df_importances.sort_values("Moyenne (%)", ascending=False).round(2).to_string())

**Lecture** : un modèle se juge sur le couple défauts détectés / précision, pas sur l'un des deux seul : détecter plus de défauts coûte en général des fausses alertes. La précision se compare au taux de défaut du train (la précision d'un tirage au hasard). Les importances ne se comparent qu'à l'intérieur d'une colonne : chaque modèle mesure le poids des variables à sa façon.

## 5. Podium et enregistrement

Le podium classe les versions retenues par la boucle de l'étape 3 par score de décision en validation, avec le détail des scores. La colonne « Écart au premier » est comparée à l'écart-type du premier : si l'écart est plus petit, le modèle est **à égalité** avec le premier (la différence est dans le bruit de la validation croisée).
Le premier du podium est enregistré dans `lab_ML/best_models/`.

In [ ]:
import joblib

premier = df_final.iloc[0]
podium = df_final[["Modèle", "Score de décision (val)", "Écart-type décision (val)", "F2 (val)", "ROC AUC (val)",
                   "Rappel (val)", "Précision (val)", "Écart train - val", "Alerte"]].copy()
podium["Écart au premier"] = premier["Score de décision (val)"] - podium["Score de décision (val)"]
podium["À égalité avec le premier"] = podium["Écart au premier"] <= premier["Écart-type décision (val)"]
print(podium.round(4).to_string(index=False))
print()
for rang, ligne in podium.head(3).iterrows():
    print(f"{['🥇', '🥈', '🥉'][rang]} {ligne['Modèle']} : score de décision {ligne['Score de décision (val)']:.4f} ± {ligne['Écart-type décision (val)']:.4f}  {ligne['Alerte']}")

# Enregistrement du meilleur modèle (version retenue par la boucle, entraînée sur tout le train par la GridSearch)
dossier_modeles = root_dir / "lab_ML" / "best_models"
dossier_modeles.mkdir(exist_ok=True)
chemin = dossier_modeles / f"model_ml_{numero_scenario}.joblib"
joblib.dump(best_models[premier["Modèle"]], chemin)
print(f"\nModèle enregistré : {chemin}")

## Conclusion

*À rédiger après l'exécution, à partir des tableaux ci-dessus, puis à reporter dans `lab_ML/tableau_ML.md`.*